## Procedural Procesing of the GMLVQ algorithm

In [17]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.datasets import load_iris, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix

In [18]:
x, y = load_breast_cancer(return_X_y= True)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

x_train, x_test = torch.tensor(x_train, dtype=torch.float32), torch.tensor(x_test, dtype=torch.float32)
y_train, y_test = torch.tensor(y_train, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32)
print(x_train.shape, x_test.shape, y_train.shape, y_test.shape, sep='|')

torch.Size([455, 30])|torch.Size([114, 30])|torch.Size([455])|torch.Size([114])


In [19]:
classes = np.unique(y)
prototypes= []
proto_labels = []
for c in classes:
    idx = (y_train == c)
    proto = x_train[idx][0]
    prototypes.append(proto)
    proto_labels.append(c)

prototypes = torch.stack(prototypes)
proto_labels = torch.tensor(proto_labels)

print('Prototypes:', prototypes, 'Classes: ', proto_labels, sep='\n')

Prototypes:
tensor([[ 0.0304,  0.9549,  0.1051, -0.1228,  0.8043,  2.7359,  1.4233,  0.4536,
          2.0996,  1.8722, -0.4102,  1.6627, -0.3504, -0.1797,  0.3403,  6.3084,
          2.7397,  0.8511,  3.7380,  3.0726, -0.1056,  1.9103, -0.0187, -0.2028,
          0.9221,  4.4516,  2.9150,  0.9617,  3.5947,  3.4342],
        [-1.1504, -0.3906, -1.1286, -0.9588,  0.3110, -0.5960, -0.8026, -0.8025,
          0.2945,  0.0943, -0.4951,  1.4872, -0.5145, -0.4915,  0.2815, -0.6045,
         -0.4690, -0.6117,  0.0580, -0.3576, -1.0432,  0.2135, -1.0360, -0.8488,
          0.3425, -0.7301, -0.8123, -0.7580, -0.0161, -0.3850]])
Classes: 
tensor([0, 1])


In [35]:
n_classes = len(classes)
n_features = x_train.shape[-1]
w = nn.Parameter(prototypes)
r_matrix = nn.Parameter(torch.ones(n_features))
lr_w = 0.01
lr_r= 0.01

original_w  = w.clone()
original_r = r_matrix.clone()

class GMLVQLoss(nn.Module):
    def forward(self, d_correct, d_wrong, EPS=1e-8):
        mu = (d_correct - d_wrong) / (d_correct + d_wrong + EPS)
        return torch.relu(mu)
        # return F.gelu(mu)


optimizer = torch.optim.Adam([
    {'params': w, 'lr': lr_w},
    {'params': r_matrix, 'lr': lr_r}
])

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=2,
    gamma=0.05
)

criterion = GMLVQLoss()

def distance(x, w, r_matrix):
    # r = torch.clamp(r_matrix, min=1e-6)
    r = torch.softmax(r_matrix, dim=0)
    diff = x.unsqueeze(0) - w
    return (r * diff * diff).sum(dim=1)


EPOCHS = 20
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for i in range(len(x_train)):
        x, label = x_train[i], y_train[i]
        distances = distance(x, w, r_matrix)
        correct_mask = label == proto_labels
        wrong_mask = ~correct_mask

        d_correct = distances[correct_mask].min()
        d_wrong = distances[wrong_mask].min() # what if i take the sum
        loss = criterion(d_correct, d_wrong)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        with torch.no_grad():
            # r_matrix.clamp(min=1e-6)
            r_matrix = torch.softmax(r_matrix, dim=0)
        epoch_loss += loss.item()
    if (epoch + 1) % 2 == 0:
        print(f'Epoch {epoch + 1}/{EPOCHS} loss={epoch_loss / len(x_train):.4f}')

Epoch 2/20 loss=0.0002
Epoch 4/20 loss=0.0002
Epoch 6/20 loss=0.0002
Epoch 8/20 loss=0.0002
Epoch 10/20 loss=0.0002
Epoch 12/20 loss=0.0002
Epoch 14/20 loss=0.0002
Epoch 16/20 loss=0.0002
Epoch 18/20 loss=0.0002
Epoch 20/20 loss=0.0002


In [33]:
@torch.no_grad()
def predict(x, w, r):
    preds = []
    for x_i in x:
        distances = distance(x_i, w, r)
        preds.append(proto_labels[distances.argmin()])
    return torch.stack(preds)

def score(x, y, w, r):
    return (predict(x, w ,r) == y).float().mean().item()

score(x_test, y_test, w, r_matrix)

0.9385964870452881

In [34]:
y_pred = predict(x_test, w, r_matrix)
y_pred

tensor([0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1,
        0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1,
        1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0,
        1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1,
        0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1])

In [30]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[46  1]
 [ 4 63]]
